In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# Exploratory Data Analysis (EDA)

### Task 1: Setup and basic inspection

In [ ]:
df = pd.read_csv("../data/processed/features_selected.csv")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.describe()

### Task 2: Univariate Analysis

In [ ]:
print(df["_BMI5"].describe()/100)
print(f"Skewness: {(df['_BMI5']/100).skew():.2f}")

In [ ]:
print((df["_BMI5"]/100 > 60).sum())
print(df[df["_BMI5"]/100 > 60]["_BMI5"].value_counts().head(10))

In [ ]:
df = df.drop(df[df["_BMI5"]>6000].index)
print(df["_BMI5"].max()/100)
print(df["_BMI5"].shape)

In [ ]:
print(type(df))
print(df.shape)

#### Unusual BMI Detection
1) I found out that 3672 rows of the BMI column had medically impossible values (like 86.51,81.37)
2) I classified these as data errors and not real BMI values
3) I decided to drop them as dropping 3672 rows out of 2.37M is negligible (0.15%) 

In [ ]:
plt.hist(df["_BMI5"]/100,bins=10,edgecolor='black')
plt.xlabel("BMI of Interviewee")
plt.ylabel("Number of Interviewees")
plt.title("BMI Distribution of Survey Respondents")
plt.axvline(df["_BMI5"].mean()/100, color='red', linestyle='--', label='Mean')
plt.legend()
plt.savefig("../reports/univariate/bmi_distribution.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
df["_BMI5_scaled"] = df["_BMI5"]/100
sns.boxplot(x='Diabetes_binary',y="_BMI5_scaled",data=df)
plt.xticks([0,1],["No Diabetes","Diabetes"])
plt.xlabel("Diabetes Binary")
plt.ylabel("BMI of the Interviewee")
plt.title("BMI Split by Diabetes")
plt.savefig("../reports/bivariate/bmi_vs_diabetes.png",bbox_inches = 'tight',dpi = 150)
plt.show()

In [ ]:
df.groupby("Diabetes_binary")["_BMI5"].median()/100

#### BMI Observation 
The median BMI for diabetic patients is noticeably higher than for non-diabetic patients, suggesting that higher BMI is associated with increased diabetes risk. This confirms BMI as a high discriminative power feature for our model. Median BMI for non-diabetic patients is 26.43 vs 30.04 for diabetic patients — a difference of 3.61 points

In [ ]:
df["EXERANY2"] = df["EXERANY2"].replace({7.0:np.nan,9.0:np.nan})
df = df.dropna(subset=["EXERANY2"])

In [ ]:
ct = pd.crosstab(df["EXERANY2"],df["Diabetes_binary"],normalize="index")
ct.plot(kind='bar',stacked=True,edgecolor = 'black')
plt.xticks([1,0],["No Exercise","Exercise"])
plt.xlabel("Participation in Exercise")
plt.ylabel("Diabetes Binary")
plt.legend(["No Diabetes","Diabetes"],bbox_to_anchor=(1.28, 1))
plt.style.use("ggplot")
plt.savefig("../reports/bivariate/diabetes_vs_exercise.png",bbox_inches = 'tight',dpi = 150)
plt.show()

### Exercise Observation
1) This stacked bar chart shows the proportion of diabetic vs non-diabetic patients among those who exercise (EXERANY2=1) and those who don't (EXERANY2=2).
2) Percentage of People who are diabetic vs non-diabetic
    - 87% of people who exercise are Not Diabetic and 13% are Diabetic
    - 77% of people who dont exercise are Not Diabetic and 23% are Diabetic
3) It was found that people who exercise have lower chance of having diabetes as compared to people who dont.

In [ ]:
df["PHYSHLTH"].value_counts()

In [ ]:
df["PHYSHLTH"] = df["PHYSHLTH"].replace({88:0,77:np.nan,99:np.nan})
df = df.dropna(subset="PHYSHLTH")
df["PHYSHLTH"].value_counts()

In [ ]:
print(df.groupby("Diabetes_binary")["PHYSHLTH"].median())
sns.boxplot(x="Diabetes_binary",y="PHYSHLTH",data=df)
plt.xticks([0,1],["No Diabetes","Diabetes"])
plt.xlabel("Diabetes Binary")
plt.ylabel("Bad Physical Health in the last 30 days")
plt.title("Impact of Physical Health on Diabetes")
plt.savefig("../reports/bivariate/physhlth_vs_diabetes.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# What % of each class reports AT LEAST ONE bad health day
df["ADDEPEV2"].value_counts()

### Physical Health (PHYSHLTH) Observation
1) PHYSHLTH is a zero-inflated feature — a large proportion of respondents 
   report 0 bad health days, which makes median alone a misleading statistic
   - Median for non-diabetic patients: 0.0 days
   - Median for diabetic patients: 1.0 days

2) A more meaningful metric is the proportion of respondents reporting 
   at least one bad health day:
   - Non-diabetic patients: 33.8% report at least one bad health day
   - Diabetic patients: 51.4% report at least one bad health day

3) This ~18 percentage point difference confirms PHYSHLTH has meaningful 
   discriminative power despite the low median difference

4) Note: Reverse causality may apply — diabetes itself causes poor physical 
   health, so the direction of this relationship cannot be confirmed from 
   survey data alone

In [ ]:
dct = pd.crosstab(df["ADDEPEV2"],df["Diabetes_binary"],normalize="index")
dct.plot(kind='bar',stacked=True,edgecolor = 'black')
plt.xticks([1,0],["Depressive Order","No Depressive Order"])
plt.xlabel("Diagnosis of Depressive Order")
plt.ylabel("Diabetes Binary")
plt.legend(["No Diabetes","Diabetes"],bbox_to_anchor=(1.28, 1))
plt.style.use("ggplot")
plt.savefig("../reports/bivariate/depressionvsdiabetes.png",bbox_inches= 'tight', dpi=150)
plt.show()

### ADDEPEV2 (Depression) Observation

1) People with depressive disorder show ~20% diabetes rate vs ~13% without

2) Clear difference between the two groups

3) Reverse causality applies — diabetes causes chronic stress, fatigue, and lifestyle restrictions which can trigger depression. Direction cannot be confirmed from survey data.

In [ ]:
cct = pd.crosstab(df["CVDINFR4"],df["Diabetes_binary"],normalize="index")
cct.plot(kind='bar',stacked=True,edgecolor = 'black')
plt.xticks([1,0],["Heart Attack","No Heart Attack"])
plt.xlabel("Diagnosis of Heart Attack")
plt.ylabel("Diabetes Binary")
plt.legend(["No Diabetes","Diabetes"],bbox_to_anchor=(1.28, 1))
plt.style.use("ggplot")
plt.savefig("../reports/bivariate/heartattackvsdiabetes.png",bbox_inches= 'tight', dpi=150)
plt.show()

### CVDINFR4 (Heart Attack) Observation

1) People with heart attack history show ~37% diabetes rate vs ~13% without

2) Strongest visual separation of all four plots — nearly 3x higher diabetes proportion

3) Metabolic syndrome links both conditions — insulin resistance drives both heart disease and diabetes
Reverse causality applies — diabetes damages blood vessels over time leading to heart attacks

In [ ]:
cvct = pd.crosstab(df["CVDCRHD4"],df["Diabetes_binary"],normalize="index")
cvct.plot(kind='bar',stacked=True,edgecolor = 'black')
plt.xticks([1,0],["Coronary Disease","No Coronary Disease"])
plt.xlabel("Diagnosis of Coronary Heart Disease")
plt.ylabel("Diabetes Binary")
plt.legend(["No Diabetes","Diabetes"],bbox_to_anchor=(1.28, 1))
plt.style.use("ggplot")
plt.savefig("../reports/bivariate/coronarydiseasevsdiabetes.png",bbox_inches= 'tight', dpi=150)
plt.show()

### CVDCRHD4 (Coronary Heart Disease) Observation

1) Nearly identical pattern to CVDINFR4 — ~37% diabetes rate in coronary disease group

2) Makes sense — heart attack and coronary disease are closely related conditions

3) Both share the same metabolic syndrome pathway

In [ ]:
df["_RFBING5"] = df["_RFBING5"].replace({1.0:0})
df["_RFBING5"] = df["_RFBING5"].replace({2.0:1})
df["_RFBING5"].value_counts()

In [ ]:
rct = pd.crosstab(df["_RFBING5"],df["Diabetes_binary"],normalize="index")
rct.plot(kind='bar',stacked=True,edgecolor = 'black')
plt.xticks([1,0],["Binge Drinking","No Binge Drinking"])
plt.xlabel("Binge Drinking Risk Factor ")
plt.ylabel("Diabetes Binary")
plt.legend(["No Diabetes","Diabetes"],bbox_to_anchor=(1.28, 1))
plt.style.use("ggplot")
#plt.savefig("../reports/bivariate/bingedrinkingvsdiabetes.png",bbox_inches= 'tight', dpi=150)
plt.show()

### _RFBING5 (Binge Drinking) Observation

1) Binge drinkers show lower diabetes rate (~7%) vs non-binge drinkers (~16%)

2) Counterintuitive result — classic reverse causality

3) Diabetic patients are medically advised to stop or reduce alcohol consumption

4) Healthy individuals are more likely to be active binge drinkers

## Day 4 — Exploratory Data Analysis

### Setup
- Loaded cleaned dataset from data/processed/features_selected.csv
- Created reports/univariate/ and reports/bivariate/ directories

### Univariate Analysis

#### BMI Distribution
- Distribution is right-skewed (skewness = 1.57)
- Majority of respondents have BMI between 20-40
- Mean BMI: ~27.8 (overweight range)
- Outliers detected above BMI 60 — 3,672 rows with impossible values 
  like 86.51, 81.37 identified as data errors and dropped (0.15% loss)
- Right skew confirms median is preferred over mean for BMI imputation

### Bivariate Analysis

#### BMI vs Diabetes (_BMI5)
- Median BMI for non-diabetic patients: 26.43 (overweight range)
- Median BMI for diabetic patients: 30.04 (obese range)
- Difference of 3.61 BMI points — clinically significant
- The 25-30 → 30+ boundary aligns with established medical literature 
  where obesity threshold marks accelerated diabetes risk
- Confirms _BMI5 as high discriminative power feature

#### Exercise vs Diabetes (EXERANY2)
- Sentinel values 7.0 and 9.0 removed before analysis
- People who don't exercise show higher diabetes proportion
- Reverse causality noted — diabetic patients are prescribed exercise 
  as part of disease management, which can inflate exercise rates 
  among diabetic respondents
- BRFSS 1/2 encoding recoded to ML convention 1/0

#### Physical Health vs Diabetes (PHYSHLTH)
- Zero-inflated feature — large proportion report 0 bad health days
- Median alone is misleading due to zero inflation:
  - Non-diabetic median: 0.0 days
  - Diabetic median: 1.0 days
- More meaningful metric — proportion reporting at least one bad health day:
  - Non-diabetic: 33.8%
  - Diabetic: 51.4%
- ~18 percentage point difference confirms discriminative power
- Reverse causality applies — diabetes causes poor physical health directly
- 88 recoded to 0 (no bad health days), 77/99 dropped as sentinels

#### Depression vs Diabetes (ADDEPEV2)
- Depressive disorder group shows ~20% diabetes rate vs ~13% without
- Meaningful separation between groups
- Reverse causality applies — chronic illness like diabetes triggers 
  depression through lifestyle restrictions and metabolic changes

#### Heart Attack vs Diabetes (CVDINFR4)
- Heart attack group shows ~37% diabetes rate vs ~13% without
- Strongest separation of all binary features analyzed
- Nearly 3x higher diabetes proportion in heart attack group
- Metabolic syndrome drives both conditions simultaneously
- Reverse causality applies — diabetic vascular damage leads to heart attacks

#### Coronary Heart Disease vs Diabetes (CVDCRHD4)
- Nearly identical pattern to CVDINFR4 (~37% diabetes rate)
- Expected — heart attack and coronary disease share the same 
  metabolic syndrome pathway
- High inter-correlation between CVDINFR4 and CVDCRHD4 suspected — 
  will verify in correlation matrix

#### Binge Drinking vs Diabetes (_RFBING5)
- Binge drinkers show lower diabetes rate (~7%) vs non-binge drinkers (~16%)
- Counterintuitive — reverse causality explains this completely
- Diabetic patients advised to reduce/stop alcohol consumption
- Encoding corrected: 1=no binge drinking → 0, 2=binge drinking → 1
- Sentinel value 9.0 removed before analysis

### Key Themes from Day 4 EDA
1. Reverse causality is present in almost every feature analyzed — 
   survey data cannot confirm direction of relationships
2. Zero-inflated features require proportion analysis, not just median
3. BMI and cardiovascular conditions show strongest visual separation 
   between diabetic and non-diabetic groups
4. All binary columns recoded from BRFSS convention (1/2) to ML 
   convention (1/0) for model compatibility

### Saved Outputs
- reports/univariate/bmi_distribution.png
- reports/bivariate/bmi_vs_diabetes.png
- reports/bivariate/diabetes_vs_exercise.png
- reports/bivariate/physhlth_vs_diabetes.png
- reports/bivariate/depression_vs_diabetes.png
- reports/bivariate/heartattack_vs_diabetes.png
- reports/bivariate/coronary_vs_diabetes.png
- reports/bivariate/binge_vs_diabetes.png

In [ ]:
binary_cols = ["EXERANY2", "ADDEPEV2","PNEUVAC3","HLTHPLN1","QLACTLM2","CHCOCNCR","HAVARTH3","HIVTST6","MEDCOST","CHCSCNCR","SEX","DRNKANY5","_HCVU651","CVDSTRK3","CHCKIDNY","CVDCRHD4","CVDINFR4"]   

In [ ]:
df_backup = df.copy()

In [ ]:
for col in binary_cols:
    sentinel_count = df[col].isin([7.0, 9.0]).sum()
    pct = sentinel_count / len(df) * 100
    print(f"{col}: {sentinel_count} sentinel values ({pct:.2f}%)")

In [ ]:
neg_binary_cols = ["ADDEPEV2", "HLTHPLN1", "QLACTLM2", "CHCOCNCR", "HAVARTH3", "MEDCOST", "CHCSCNCR", "CVDSTRK3", "CHCKIDNY", "CVDCRHD4", "CVDINFR4"]
df[neg_binary_cols] = df[neg_binary_cols].replace({7.0:np.nan,9.0:np.nan})
df = df.dropna(subset=neg_binary_cols)

In [ ]:
sig_binary_cols = ["PNEUVAC3", "HIVTST6", "DRNKANY5", "_HCVU651"]
df[sig_binary_cols] = df[sig_binary_cols].replace({7.0: np.nan, 9.0: np.nan})
imputer = SimpleImputer(strategy="most_frequent")
impute = pd.DataFrame(df[["PNEUVAC3", "HIVTST6", "DRNKANY5", "_HCVU651"]])
df[["PNEUVAC3", "HIVTST6", "DRNKANY5", "_HCVU651"]] = imputer.fit_transform(impute)


In [ ]:
df["PERSDOC2"] = df["PERSDOC2"].replace({7.0:np.nan,9.0:np.nan})
df = df.dropna(subset="PERSDOC2")
df["PERSDOC2"] = df["PERSDOC2"].replace({2.0:1,3.0:0})

In [ ]:
df[binary_cols] = df[binary_cols].replace({2.0:0})

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns="_BMI5_scaled")

In [ ]:
print(df.isnull().sum().sum())

In [ ]:
null_columns = [col for col in df.columns if df[col].isnull().sum() !=0 ]
null_columns

In [ ]:
df[null_columns].isnull().sum()

In [ ]:
for col in ["_BMI5", "ALCDAY5", "INCOME2", "EDUCA"]:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))
    print(f"Null count: {df[col].isnull().sum()}")

In [ ]:
df["ALCDAY5"].isnull().sum()

In [ ]:
df["ALCDAY5"].value_counts()

In [ ]:
df = df.dropna(subset=["INCOME2","EDUCA"])
med_imputer = SimpleImputer(strategy="median")
med_impute = pd.DataFrame(df["_BMI5"])
df["_BMI5"] = med_imputer.fit_transform(med_impute)
df["ALCDAY5"] = df["ALCDAY5"].replace({888:0})
df["ALCDAY5"] = df["ALCDAY5"].replace({777.0:np.nan,999.0:np.nan})
df = df.dropna(subset="ALCDAY5")

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sum()

In [ ]:
print(df.shape)
df = df.drop(columns=['_AGE_G','_AGE65YR'])
print(df.shape)

In [ ]:
corr_with_target = df.corr()["Diabetes_binary"].sort_values(ascending=False).round(decimals=2).head(14)
corr_with_target_df = pd.DataFrame(corr_with_target)
top_features = corr_with_target.index.tolist()
top_features = [f for f in top_features if f!="Diabetes_binary"]
corr_matrix = df[top_features].corr()

In [ ]:
corr_with_target_df = corr_with_target_df.drop(corr_with_target_df.index[0])

In [ ]:
sns.heatmap(corr_with_target_df,annot=True,cmap='coolwarm')
plt.title("Correlation of features with Diabetes binary")
plt.savefig("../reports/bivariate/Correlationwithdiabetes.png",bbox_inches = 'tight',dpi=150)
plt.show()

In [ ]:
sns.heatmap(corr_matrix,annot=True,cmap='RdBu_r')
plt.title("Correlation Matrix of features")
plt.savefig("../reports/bivariate/Correlationoffeatures.png",bbox_inches = 'tight',dpi=150)
plt.show()

In [ ]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)

## Day 5 — Binary Column Encoding, Data Cleaning & Correlation Analysis

### 1) Binary Column Encoding
- Identified 17 binary columns following BRFSS 1/2 convention
- Categorized columns into two groups based on sentinel value percentage:
  - **Negligible (<1%):** ADDEPEV2, HLTHPLN1, QLACTLM2, CHCOCNCR, HAVARTH3, 
    MEDCOST, CHCSCNCR, CVDSTRK3, CHCKIDNY, CVDCRHD4, CVDINFR4
    → Replaced 7.0/9.0 with NaN and dropped rows
  - **Significant (>1%):** PNEUVAC3 (8.44%), HIVTST6 (3.04%), 
    DRNKANY5 (4.53%), _HCVU651 (33.79%)
    → Replaced 7.0/9.0 with NaN and imputed using mode (most_frequent)
- _HCVU651 had 33.79% sentinel values — retained as healthcare coverage 
  is clinically important and missingness likely reflects MNAR bias 
  (shame/negligence in reporting)
- PERSDOC2 was a 3-category column (1=one doctor, 2=more than one, 3=no doctor)
  → Recoded to binary: 1/2 → 1 (has doctor), 3 → 0 (no doctor)
- All binary columns recoded from BRFSS convention (1/2) to ML convention (1/0)

### 2) Remaining Null Value Treatment
- **_BMI5** (5.6% missing) → Imputed with median (preferred over mean 
  due to right-skewed distribution, skewness=1.57)
- **ALCDAY5** → 888 recoded to 0 (no drinks), 777/999 dropped as sentinels
- **INCOME2** (0.4%) and **EDUCA** (0.1%) → Rows dropped (negligible loss)
- Redundant age columns **_AGE_G** and **_AGE65YR** dropped — 
  keeping _AGEG5YR (13 age bands, highest information content)

### 3) Final Dataset Shape
- Rows: 2,120,457
- Columns: 37
- Null values: 0

### 4) Correlation Analysis — Features vs Diabetes_binary
Top correlated features with target variable:

| Feature | Correlation | Interpretation |
|---|---|---|
| GENHLTH | 0.27 | General health — strongest predictor |
| _BMI5 | 0.23 | Higher BMI strongly associated with diabetes |
| PNEUVAC3 | 0.19 | Pneumonia vaccine — older/sicker population proxy |
| _AGEG5YR | 0.19 | Older age groups show higher diabetes prevalence |
| _RFHLTH | 0.18 | Poor reported health |
| PHYSHLTH | 0.17 | More bad physical health days |
| HAVARTH3 | 0.16 | Arthritis — shared inflammation pathway |
| QLACTLM2 | 0.16 | Activity limitation due to health |

### 5) Inter-Feature Correlation (Multicollinearity Check)
- No feature pairs exceeded the 0.7 multicollinearity threshold
- Closest pair: **_RFHLTH and GENHLTH at 0.69** — borderline, 
  will monitor during model training
- If either degrades model performance, _RFHLTH will be dropped 
  as GENHLTH has higher correlation with target

### 6) Key Limitations Noted
- **Reverse causality** — GENHLTH may be low because of diabetes, 
  not a cause of it
- **MNAR bias** — _HCVU651 missingness is not random, 
  likely systematic underreporting
- **Survey data** — correlations cannot imply causation

### 7) Saved Outputs
- Cleaned dataset: `data/processed/cleaned_data.csv`
- Correlation with target heatmap: `reports/bivariate/Correlationwithdiabetes.png`
- Inter-feature correlation matrix: `reports/bivariate/Correlationoffeatures.png`

### 8) PNEUVAC3 — Reverse Causality Warning
PNEUVAC3 shows 0.19 correlation with Diabetes_binary but this is 
likely a consequence proxy — diabetic patients are prescribed pneumonia 
vaccines because of their condition. For a real-world risk prediction 
tool targeting undiagnosed individuals, this feature may introduce 
misleading signals and will be reviewed before final model training.

## Final Sentinel Value Audit

In [ ]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [ ]:
df_backup = df.copy()

In [ ]:
sentinel_vals = [7.0, 9.0, 77.0, 99.0, 777.0, 999.0, 7777.0, 9999.0]
for col in df.columns:
    count = df[col].isin(sentinel_vals).sum()
    if count > 0:
        print(f"{col}: {count} sentinel values")

In [ ]:
print(sorted(df["_AGEG5YR"].unique()))

In [ ]:
df["INCOME2"] = df["INCOME2"].replace({77.0:np.nan,99.0:np.nan})

In [ ]:
imputer = SimpleImputer(strategy="most_frequent")
impute = pd.DataFrame(df["INCOME2"])
df["INCOME2"] = imputer.fit_transform(impute)


In [ ]:
df = df.drop(columns=["DROCDY3_"])

In [ ]:
sentinel_vals = [7.0, 9.0, 77.0, 99.0, 777.0, 999.0, 7777.0, 9999.0]
for col in df.columns:
    count = df[col].isin(sentinel_vals).sum()
    if count > 0:
        print(f"{col}: {count} sentinel values")

In [ ]:
df["_ASTHMS1"].value_counts()

In [ ]:
df["MENTHLTH"] = df["MENTHLTH"].replace({88.0:0})
df["CHECKUP1"] = df["CHECKUP1"].replace({8.0:0})
df["CHECKUP1"].value_counts()

In [ ]:
df["_AIDTST3"] = df["_AIDTST3"].replace({9.0:np.nan})
df["MENTHLTH"] = df["MENTHLTH"].replace({77.0:np.nan,99.0:np.nan})

In [ ]:
simpute = pd.DataFrame(df[["_AIDTST3","MENTHLTH"]])
df[["_AIDTST3","MENTHLTH"]] = imputer.fit_transform(simpute)

In [ ]:
print(df.shape)
print(df.isnull().sum().sum())

In [ ]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)

## Final Recoding Audit

In [ ]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [ ]:
df["_DRDXAR1"] = df["_DRDXAR1"].replace({2.0:0})
df["_AIDTST3"] = df["_AIDTST3"].replace({2.0:0})
df["_RFHLTH"] = df["_RFHLTH"].replace({2.0:0})
df["_TOTINDA"] = df["_TOTINDA"].replace({2.0:0})

In [ ]:
print(df["_DRDXAR1"].value_counts())
print(df["_AIDTST3"].value_counts())
print(df["_RFHLTH"].value_counts())
print(df["_TOTINDA"].value_counts())

In [ ]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)